In [34]:
#Importing the dataset
import pandas as pd
dataset=pd.read_csv("customer_shopping_behavior.csv")

In [4]:
dataset.2

SyntaxError: invalid syntax (157599890.py, line 1)

In [36]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [37]:
print(dataset['Payment Method'][:15])

0             Venmo
1              Cash
2       Credit Card
3            PayPal
4            PayPal
5             Venmo
6              Cash
7       Credit Card
8             Venmo
9              Cash
10    Bank Transfer
11    Bank Transfer
12            Venmo
13           PayPal
14       Debit Card
Name: Payment Method, dtype: object


In [38]:
dataset.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [39]:
dataset.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [40]:
''' Now filling the only null values present in Review Ratings(with the median values),
because mean won't help here for better analysis.'''

dataset['Review Rating'] = dataset.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [41]:
dataset.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [42]:
##snake casing better analysis

dataset.columns=dataset.columns.str.lower()
dataset.columns=dataset.columns.str.replace(' ','_')

In [43]:
dataset=dataset.rename(columns={'purchase_amount_(usd)':'purchase_amount'})
dataset.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [44]:
#now we are going to split the sells as per age by creating a column(quantile cutting)
labels=['Young','Adult','Mid_age','Senior']
dataset['age_group']=pd.qcut(dataset['age'],q=4,labels=labels)
print(dataset[['age_group','age']].head())
print(dataset['age_group'].value_counts())

  age_group  age
0   Mid_age   55
1     Young   19
2   Mid_age   50
3     Young   21
4   Mid_age   45
age_group
Young      1028
Mid_age     986
Senior      944
Adult       942
Name: count, dtype: int64


By categorizing the ages we have got 1028 young, 942 adults, 986 mid-aged and 944 senior customers.

In [45]:
dataset['frequency_of_purchases'].unique()

array(['Fortnightly', 'Weekly', 'Annually', 'Quarterly', 'Bi-Weekly',
       'Monthly', 'Every 3 Months'], dtype=object)

In [46]:
##Creating a column for transforming frequency_of_purchases column val---> numerical val. 
mapping_frequencies_of_purchases={
    'Weekly': 7,
    'Bi-Weekly': 14,
    'Fortnightly': 14,
    'Monthly': 30,
    'Quarterly': 90,
    'Every 3 Months': 90,
    'Annually': 365
      
}
dataset['purchasing_frequency']=dataset['frequency_of_purchases'].map(mapping_frequencies_of_purchases)


In [47]:
dataset[['purchasing_frequency','frequency_of_purchases']].head(8)

,purchasing_frequency,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly


Now checking for the redundant columns


In [48]:
(dataset['promo_code_used']==dataset['discount_applied']).all()

True

In [49]:
dataset=dataset.drop('promo_code_used',axis=1)

In [50]:
dataset.shape

(3900, 19)

In [51]:
pip install psycopg2-binary sqlalchemy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
pip install urllib3

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
#Connecting with postgresql sever
username = 'postgres'
password = quote_plus('2025@postgre')
host = 'localhost'
port = '5432' 
database = 'Reatil_Fashion_Shop_Customer_Behaviour_Analysis'

engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

#Load dataframe into sql server
table_name='customer'
dataset.to_sql(table_name, engine, if_exists='replace',index=False)

print(f"Data Successfully loaded into {table_name} in database {database}.")


Data Successfully loaded into customer in database Reatil_Fashion_Shop_Customer_Behaviour_Analysis.
